# 02 - Preprocessing
This notebook demonstrates normalization and augmentation behavior.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import tensorflow as tf
from src.data_loader import DataLoader
from src import config

loader = DataLoader()
train_gen, val_gen, test_gen = loader.get_generators()

In [ ]:
batch_x, batch_y = next(train_gen)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(batch_x[i])
    ax.set_title(f"Label: {int(batch_y[i])}")
    ax.axis("off")
plt.tight_layout(); plt.savefig("../results/preprocessing_augmented_batch.png", dpi=180)

In [ ]:
raw_datagen = tf.keras.preprocessing.image.ImageDataGenerator()
raw_gen = raw_datagen.flow_from_directory(
    directory=str(config.TRAIN_DIR), target_size=(config.IMG_SIZE, config.IMG_SIZE),
    batch_size=1, class_mode="binary", shuffle=False
)
aug_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255, rotation_range=15, width_shift_range=0.1,
    height_shift_range=0.1, horizontal_flip=True, zoom_range=0.1
)
aug_gen = aug_datagen.flow_from_directory(
    directory=str(config.TRAIN_DIR), target_size=(config.IMG_SIZE, config.IMG_SIZE),
    batch_size=1, class_mode="binary", shuffle=False
)
raw_img, _ = next(raw_gen)
fig, axes = plt.subplots(1, 4, figsize=(12, 4))
axes[0].imshow(raw_img[0].astype("uint8")); axes[0].set_title("Original")
axes[0].axis("off")
for i in range(1,4):
    aug_img, _ = next(aug_gen)
    axes[i].imshow(aug_img[0])
    axes[i].set_title(f"Augmented {i}")
    axes[i].axis("off")
plt.tight_layout(); plt.savefig("../results/preprocessing_original_vs_augmented.png", dpi=180)

## Notes
- Images are normalized via `rescale=1/255`.
- Training-only augmentation improves generalization and reduces overfitting risk.
- Validation and test generators use only rescaling to preserve evaluation integrity.